In [ ]:
# M1 V3 (alpha = float*) - 1/4: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
!nvidia-smi
r=subprocess.run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mono-complete', shell=True, capture_output=True, text=True, timeout=1200)
print('mono install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('mono --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# M1 V2 - 2/4: clone backend + symlink .dll -> thuvien .so cua torch (khong bien dich gi)
import os, subprocess, glob
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
roots=['/usr/local/cuda/lib64','/usr/local/cuda*/lib64','/usr/lib/x86_64-linux-gnu']
def findlib(nm):
    out=[]
    for r in roots:
        out+=glob.glob(r+'/'+nm+'.so*')
    return [p for p in out if os.path.exists(os.path.realpath(p))]
cudart_c=sorted(findlib('libcudart'))
cublas_c=sorted(findlib('libcublas'))
print('cudart', cudart_c[:3]); print('cublas', cublas_c[:3])
assert cudart_c and cublas_c
libdir='/content/tkvlib'; os.makedirs(libdir, exist_ok=True)
for nm,tgt in [('cudart.dll',cudart_c[0]),('libcudart.dll',cudart_c[0]),('cublas.dll',cublas_c[0]),('libcublas.dll',cublas_c[0])]:
    p=libdir+'/'+nm
    if os.path.lexists(p): os.remove(p)
    os.symlink(tgt,p)
    print('symlink', nm, '->', tgt)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH=', os.environ['LD_LIBRARY_PATH'][:200])

In [ ]:
# M1 - 3/4: chay PE do .tkv -> IL (khong co .cpp/.cu trong duong nay)
import subprocess, os, re
exe=d+'/Release/cuda_backend/examples/ilgpu_saxpy.exe'
print('exe exists=',os.path.exists(exe),' bytes=',os.path.getsize(exe) if os.path.exists(exe) else 0)
print('co file C++ sinh?', [p for p in ['ilgpu_saxpy.torch.cpp','ilgpu_saxpy.cuda.cu'] if os.path.exists(os.path.dirname(exe)+'/'+p)])
r=subprocess.run(['mono',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono rc=', r.returncode)
print(r.stdout)
print('STDERR:', (r.stderr or '')[:2000])
m=re.search(r'MAXERR\s+([0-9.eE+-]+)', r.stdout)
print('PARSED MAXERR=', m.group(1) if m else 'KHONG THAY')

In [ ]:
# M2 - chay SGEMM (IL-only, khong torch)
import subprocess, os, re
exe2=d+'/Release/cuda_backend/examples/ilgpu_sgemm.exe'
print('sgemm exe exists=',os.path.exists(exe2),' bytes=',os.path.getsize(exe2) if os.path.exists(exe2) else 0)
print('co file C++ sinh?', [p for p in ['ilgpu_sgemm.torch.cpp','ilgpu_sgemm.cuda.cu'] if os.path.exists(os.path.dirname(exe2)+'/'+p)])
r2=subprocess.run(['mono',exe2], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono sgemm rc=', r2.returncode)
print(r2.stdout)
print('STDERR:', (r2.stderr or '')[:1500])
m2=re.search(r'MAXERR\s+([0-9.eE+-]+)', r2.stdout)
print('PARSED SGEMM MAXERR=', m2.group(1) if m2 else 'KHONG THAY')

In [ ]:
# Ket luan khong torch
print('SAXPY  MAXERR =', m.group(1) if m else 'KHONG THAY')
print('SGEMM  MAXERR =', m2.group(1) if m2 else 'KHONG THAY')
ok = bool(m and m2 and float(m.group(1))<1e-4 and float(m2.group(1))<1e-4)
print('KET LUAN IL-ONLY (khong torch):', 'OK' if ok else 'XEM OUTPUT')